# Prompt 13 — Held-out Evaluation (real outputs only)
**Study:** Token-level uncertainty as a predictor of hallucination. **Frozen choice:** p90 NLL (selected on validation AP). **Primary metric:** Average Precision (AP). **Secondary:** AUROC. The test set is read **only after** protocol and validation consistency checks. Run each cell and retain its real output; this notebook contains no fabricated results.

Upload **both** `AITR_Prompt10_FINAL.zip` and `AITR_Prompt12_outputs.zip`. No GPU is required.

In [ ]:
import io, json, zipfile, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, roc_curve
from google.colab import files
print('Upload AITR_Prompt10_FINAL.zip AND AITR_Prompt12_outputs.zip')
uploaded = files.upload()
required_zips = ['AITR_Prompt10_FINAL.zip', 'AITR_Prompt12_outputs.zip']
assert all(Path(x).exists() for x in required_zips), f'Missing required zip(s): {[x for x in required_zips if not Path(x).exists()]}'
with zipfile.ZipFile(required_zips[0]) as z:
    data = {s: pd.read_csv(io.BytesIO(z.read(f'{s}_scores.csv'))) for s in ['train','validation','test']}
    validation_report = json.loads(z.read('baseline_validation.json'))
with zipfile.ZipFile(required_zips[1]) as z:
    protocol = json.loads(z.read('frozen_comparison_protocol.json'))
    val_record = pd.read_csv(io.BytesIO(z.read('validation_comparison.csv')))
print('Loaded split sizes:', {s:len(df) for s,df in data.items()})
print('Frozen method:', protocol['selected_single_signal'])

## 1. Preflight checks — no test metric computed here
Check columns, numeric scores, label semantics, split sizes, ID uniqueness, no repeated row positions, and that saved validation metrics reproduce Prompt 12. Synthetic IDs do **not** independently establish source-record disjointness.

In [ ]:
assert protocol['selected_single_signal'] == 'p90_nll', 'Frozen choice changed: stop and investigate'
assert protocol['primary_metric'] == 'average_precision'
assert protocol['positive_label'] == 1
assert protocol['score_direction'] == 'higher means more predicted hallucination'
assert protocol['test_set_status'].startswith('NOT EVALUATED')
assert validation_report['checks']['score_integrity'] == 'PASS'
assert validation_report['checks']['id_integrity'] == 'PASS'
cols = ['ID','row_index','label','status','mean_nll','p90_nll','answer_token_length']
for s,df in data.items():
    assert len(df) == protocol['splits'][s], f'{s}: wrong row count'
    assert set(cols).issubset(df.columns), f'{s}: missing columns'
    assert df.ID.notna().all() and df.ID.astype(str).is_unique
    assert df.row_index.is_unique and set(df.row_index.astype(int)) == set(range(len(df)))
    assert df.status.eq('ok').all()
    assert set(df.label.dropna().astype(int).unique()).issubset({0,1})
    assert df.label.notna().all()
    assert np.isfinite(df[['mean_nll','p90_nll','answer_token_length']].to_numpy(dtype=float)).all()
    assert (df[['mean_nll','p90_nll']].to_numpy(dtype=float)>=0).all()
    assert (df.answer_token_length > 0).all()
    assert df.ID.astype(str).str.startswith(s+'_').all(), f'{s}: unexpected ID prefix'
assert pd.concat([d.ID for d in data.values()], ignore_index=True).is_unique
assert set(data['validation'].label.astype(int)) == {0,1}
assert set(data['test'].label.astype(int)) == {0,1}
methods = {'Mean token NLL (baseline)':'mean_nll',
           '90th percentile token NLL (proposed)':'p90_nll',
           'Answer length (control)':'answer_token_length'}
val = data['validation']
for name,col in methods.items():
    record=val_record.loc[val_record.column==col]
    assert len(record)==1
    ap=average_precision_score(val.label.astype(int), val[col])
    auc=roc_auc_score(val.label.astype(int), val[col])
    assert np.isclose(ap,float(record.iloc[0].average_precision),atol=1e-10)
    assert np.isclose(auc,float(record.iloc[0].auroc),atol=1e-10)
print('=== PREFLIGHT / VALIDATION REPRODUCTION: PASS ===')
print('Note: Original-source ID overlap cannot be independently audited from synthetic IDs.')

## 2. One-time held-out test evaluation
The **same 90 rows and labels** are used for all three signals. Larger scores indicate a higher predicted hallucination risk. No test-based threshold tuning.

In [ ]:
test=data['test'].sort_values('row_index').reset_index(drop=True)
y=test.label.astype(int).to_numpy()
results=[]
for name,col in methods.items():
    score=test[col].astype(float).to_numpy()
    results.append({'method':name,'column':col,'split':'test','n':len(y),
                    'positives':int(y.sum()),'prevalence':float(y.mean()),
                    'average_precision':float(average_precision_score(y,score)),
                    'auroc':float(roc_auc_score(y,score))})
result_df=pd.DataFrame(results)
print('=== PROMPT 13 HELD-OUT TEST RESULTS ===')
print(result_df.to_string(index=False, float_format=lambda x:f'{x:.6f}'))
base=float(result_df.loc[result_df.column=='mean_nll','average_precision'].iloc[0])
p90=float(result_df.loc[result_df.column=='p90_nll','average_precision'].iloc[0])
print('P90 minus mean NLL test AP (descriptive):',round(p90-base,6))

## 3. Figures — PR and ROC curves
AP is the primary metric because hallucinations are the minority class. PR baseline is test prevalence. Curves and figures are generated **from real test rows**.

In [ ]:
out=Path('/content/AITR_Prompt13_results');out.mkdir(parents=True,exist_ok=True)
result_df.to_csv(out/'results.csv',index=False)
fig,ax=plt.subplots(figsize=(7,5))
for name,col in methods.items():
    prec,rec,_=precision_recall_curve(y,test[col].to_numpy(dtype=float))
    ap=average_precision_score(y,test[col])
    ax.plot(rec,prec,label=f'{name} (AP={ap:.3f})')
ax.axhline(y.mean(),linestyle='--',color='gray',label=f'Prevalence={y.mean():.3f}')
ax.set(xlabel='Recall',ylabel='Precision',title='Held-out test: Precision–Recall');ax.legend(fontsize=8);ax.grid(alpha=.25)
fig.tight_layout();fig.savefig(out/'test_precision_recall.png',dpi=220);fig.savefig(out/'test_precision_recall.pdf');plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,5))
for name,col in methods.items():
    fpr,tpr,_=roc_curve(y,test[col].to_numpy(dtype=float))
    auc=roc_auc_score(y,test[col])
    ax.plot(fpr,tpr,label=f'{name} (AUROC={auc:.3f})')
ax.plot([0,1],[0,1],'--',color='gray',label='Chance')
ax.set(xlabel='False positive rate',ylabel='True positive rate',title='Held-out test: ROC');ax.legend(fontsize=8);ax.grid(alpha=.25)
fig.tight_layout();fig.savefig(out/'test_roc.png',dpi=220);fig.savefig(out/'test_roc.pdf');plt.show();plt.close(fig)
print('=== FIGURES SAVED ===')

## 4. Deeper analysis: paired bootstrap and error examples
A **paired** bootstrap resamples rows jointly for both methods. Percentile intervals are exploratory (small sample, no claim of significance). Failure-case table identifies ranking disagreements, but source texts are absent from the score ZIP, so examples can only be identified by synthetic row ID.

In [ ]:
rng=np.random.default_rng(42)
mean_score=test.mean_nll.to_numpy(dtype=float)
p90_score=test.p90_nll.to_numpy(dtype=float)
diffs=[]
for _ in range(2000):
    idx=rng.integers(0,len(y),size=len(y))
    if np.unique(y[idx]).size<2:continue
    diffs.append(average_precision_score(y[idx],p90_score[idx])-average_precision_score(y[idx],mean_score[idx]))
ci=np.percentile(diffs,[2.5,97.5])
print('=== PAIRED BOOTSTRAP (EXPLORATORY) ===')
print('P90 - mean AP:',round(p90-base,6),'95% percentile interval:',np.round(ci,6),'valid replicates:',len(diffs))
rank_mean=pd.Series(mean_score).rank(pct=True).to_numpy()
rank_p90=pd.Series(p90_score).rank(pct=True).to_numpy()
errors=test[['ID','row_index','label','mean_nll','p90_nll','answer_token_length']].copy()
errors['mean_percentile_rank']=rank_mean
errors['p90_percentile_rank']=rank_p90
errors['rank_disagreement']=np.abs(rank_p90-rank_mean)
errors.sort_values('rank_disagreement',ascending=False).head(15).to_csv(out/'rank_disagreement_examples.csv',index=False)
print('Top ranking disagreements (not necessarily model classification errors):')
print(errors.sort_values('rank_disagreement',ascending=False).head(8).to_string(index=False))

## 5. Length sensitivity (exploratory)
Split test responses by the **validation-set median answer length** (fixed without viewing test outcomes). Group-specific AP depends on group prevalence; compare within groups rather than treating across-group AP as a controlled effect.

In [ ]:
cutoff=float(data['validation'].answer_token_length.median())
length_rows=[]
for name,mask in [('short',test.answer_token_length<=cutoff),('long',test.answer_token_length>cutoff)]:
    subset=test.loc[mask]
    yy=subset.label.astype(int).to_numpy()
    for method,col in methods.items():
        length_rows.append({'group':name,'validation_median_cutoff':cutoff,'method':method,'n':len(subset),
                            'positives':int(yy.sum()),'prevalence':float(yy.mean()) if len(yy) else np.nan,
                            'average_precision':float(average_precision_score(yy,subset[col])) if len(yy) and yy.sum()>0 else np.nan,
                            'auroc':float(roc_auc_score(yy,subset[col])) if len(np.unique(yy))==2 else np.nan})
length_df=pd.DataFrame(length_rows)
length_df.to_csv(out/'test_length_sensitivity.csv',index=False)
print('=== TEST LENGTH SENSITIVITY (EXPLORATORY) ===')
print(length_df.to_string(index=False))

## 6. Audit and export
The source-score files contain no original query/response text or original HaluEval IDs. Therefore exact source-level overlap and text-level error interpretation remain **unverified**, not silently treated as passed. The test set was used for reporting, not method selection.

In [ ]:
audit={'prompt':13,'dataset':'HaluEval general_data, 600 sampled rows',
       'model':'distilgpt2 surrogate scorer','positive_label':1,
       'selection_source':'Prompt 12 validation AP','frozen_selected_signal':protocol['selected_single_signal'],
       'primary_metric':'average_precision','secondary_metric':'auroc',
       'split_counts':{k:len(v) for k,v in data.items()},
       'test_positive_count':int(y.sum()),'test_prevalence':float(y.mean()),
       'preflight':'PASS','score_file_integrity':'PASS',
       'source_record_overlap':'NOT VERIFIED (synthetic split IDs)',
       'original_row_audit':protocol.get('original_row_audit','NOT PERFORMED'),
       'test_threshold_tuning':'NONE','paired_bootstrap_replicates':len(diffs),
       'bootstrap_ap_difference_ci_95':list(map(float,ci)),
       'bootstrap_note':'Exploratory paired row bootstrap; not a statistical-significance claim',
       'validation_median_length_cutoff':cutoff}
(out/'evaluation_audit.json').write_text(json.dumps(audit,indent=2))
summary=['# Prompt 13 — Actual held-out evaluation','',
         'Selected on validation: p90_nll. Primary metric: average precision.','',
         '## Test metrics','',result_df.to_markdown(index=False,floatfmt='.5f'),'',
         f'P90 minus mean AP: {p90-base:+.5f}. Exploratory paired bootstrap 95% interval: [{ci[0]:.5f}, {ci[1]:.5f}].','',
         '## Caveats','- Original source IDs and texts were not retained in score CSVs; source-level disjointness and text error analysis remain unverified.',
         '- DistilGPT-2 scores are surrogate uncertainty for answers generated by a different model.',
         '- The held-out test contains only 90 examples; bootstrap intervals are exploratory.',
         '- No test-based method selection or threshold tuning was performed.']
(out/'evaluation_summary.md').write_text('\n'.join(summary)+'\n')
zip_out='/content/AITR_Prompt13_results.zip'
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):z.write(f,arcname=f.name)
print('=== PROMPT 13 COMPLETE ===')
print('Saved:',zip_out)
files.download(zip_out)